# 14 — Fraud Classification Models

## Projet : AI Business Advisor V2

### Objectif

Cette étape vise à comparer plusieurs modèles supervisés pour détecter
les déclarations potentiellement frauduleuses.

La cible est :

FraudFound_P

avec :

- 0 : non fraude ;
- 1 : fraude.

Le dataset est fortement déséquilibré, avec environ 6 % de fraudes.

La métrique principale sera donc la PR-AUC.

Les métriques secondaires sont :

- ROC-AUC ;
- Precision ;
- Recall ;
- F1-score ;
- Brier Score.

Le protocole est temporel :

- 1994 : entraînement ;
- 1995 : validation ;
- 1996 : test final.

L'année 1996 ne sera pas utilisée pendant la sélection des modèles.

In [1]:
import pandas as pd
import numpy as np

import json
import time

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.dummy import DummyClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix
)

from xgboost import XGBClassifier


RANDOM_STATE = 42

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.width",
    200
)

In [2]:
cwd = Path.cwd()

possible_processed_paths = [
    cwd / "data" / "processed",
    cwd / "data-science" / "data" / "processed",
    cwd.parent / "data" / "processed"
]

PROCESSED_DIR = None

for path in possible_processed_paths:

    if path.exists():

        PROCESSED_DIR = path
        break


if PROCESSED_DIR is None:

    raise FileNotFoundError(
        "Impossible de localiser data/processed"
    )


DATA_DIR = (
    PROCESSED_DIR.parent
)

METADATA_DIR = (
    DATA_DIR
    / "metadata"
)

REPORT_DIR = (
    DATA_DIR.parent
    / "reports"
    / "models"
    / "fraud"
    / "initial"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


fraud = pd.read_csv(
    PROCESSED_DIR
    / "fraud_modeling_base.csv"
)


print(
    "Fraud dataset :",
    fraud.shape
)

Fraud dataset : (15420, 25)


In [7]:
print("=== CLÉS DU FEATURE MANIFEST ===")

for key in feature_manifest.keys():
    print("-", key)

print("\n=== CONTENU COMPLET ===")
print(
    json.dumps(
        feature_manifest,
        indent=4,
        ensure_ascii=False
    )
)

=== CLÉS DU FEATURE MANIFEST ===
- claim_occurrence
- claim_frequency
- fraud_detection

=== CONTENU COMPLET ===
{
    "claim_occurrence": {
        "target": "has_claim",
        "numeric_features": [
            "Exposure",
            "VehPower",
            "VehAge",
            "DrivAge",
            "BonusMalus",
            "log_density"
        ],
        "categorical_features": [
            "Area",
            "VehBrand",
            "VehGas",
            "Region"
        ],
        "excluded_features": [
            "IDpol",
            "ClaimNb"
        ]
    },
    "claim_frequency": {
        "target": "ClaimNb",
        "exposure": "Exposure",
        "numeric_features": [
            "VehPower",
            "VehAge",
            "DrivAge",
            "BonusMalus",
            "log_density"
        ],
        "categorical_features": [
            "Area",
            "VehBrand",
            "VehGas",
            "Region"
        ],
        "excluded_features": [
        

In [8]:
# ============================================================
# FRAUD FEATURE CONFIGURATION
# ============================================================

with open(
    METADATA_DIR / "feature_manifest.json",
    "r",
    encoding="utf-8"
) as f:
    feature_manifest = json.load(f)


print(
    "Feature manifest keys :",
    list(feature_manifest.keys())
)


# ============================================================
# CAS 1 : configuration Fraud déjà présente
# ============================================================

if "fraud" in feature_manifest:

    print(
        "\n✅ Configuration 'fraud' trouvée dans le manifest."
    )

    fraud_config = (
        feature_manifest["fraud"]
    )


# ============================================================
# CAS 2 : configuration Fraud absente
# ============================================================

else:

    print(
        "\n⚠️ Clé 'fraud' absente du feature_manifest.json."
    )

    print(
        "Reconstruction explicite selon les décisions "
        "de Feature Engineering."
    )


    # --------------------------------------------------------
    # Variables continues
    # --------------------------------------------------------

    continuous_features = [
        "Age",
        "Deductible"
    ]


    # --------------------------------------------------------
    # Variables ordinales / numériques discrètes
    # --------------------------------------------------------

    ordinal_features = [
        "WeekOfMonth",
        "WeekOfMonthClaimed",
        "DriverRating"
    ]


    # --------------------------------------------------------
    # Colonnes exclues de la modélisation
    # --------------------------------------------------------

    excluded_features = {

        # Identifiant
        "PolicyNumber",

        # Target
        "FraudFound_P",

        # Variable temporelle utilisée pour le split
        "Year",

        # Proxy / review variable
        "RepNumber",

        # Redondante avec VehicleCategory + BasePolicy
        "PolicyType",

        # Variables potentiellement indisponibles
        # au moment réel de la prédiction
        "Fault",
        "PoliceReportFiled",
        "WitnessPresent",
        "NumberOfSuppliments",
        "Days_Policy_Accident",
        "Days_Policy_Claim",
        "AddressChange_Claim"
    }


    # --------------------------------------------------------
    # Vérifier que les features continues / ordinales existent
    # --------------------------------------------------------

    missing_continuous = [
        col
        for col in continuous_features
        if col not in fraud.columns
    ]

    missing_ordinal = [
        col
        for col in ordinal_features
        if col not in fraud.columns
    ]


    if missing_continuous:

        raise ValueError(
            f"Continuous features absentes : "
            f"{missing_continuous}"
        )


    if missing_ordinal:

        raise ValueError(
            f"Ordinal features absentes : "
            f"{missing_ordinal}"
        )


    # --------------------------------------------------------
    # Toutes les autres variables autorisées deviennent
    # des variables catégorielles
    # --------------------------------------------------------

    categorical_features = [
        col
        for col in fraud.columns
        if (
            col not in excluded_features
            and col not in continuous_features
            and col not in ordinal_features
        )
    ]


    # --------------------------------------------------------
    # Configuration finale
    # --------------------------------------------------------

    fraud_config = {

        "target":
            "FraudFound_P",

        "split_column":
            "Year",

        "continuous_features":
            continuous_features,

        "ordinal_features":
            ordinal_features,

        "categorical_features":
            categorical_features,

        "excluded_features":
            sorted(
                list(
                    excluded_features
                )
            )
    }


# ============================================================
# EXTRACTION DES GROUPES
# ============================================================

continuous_features = (
    fraud_config.get(
        "continuous_features",
        fraud_config.get(
            "numeric_features",
            []
        )
    )
)


ordinal_features = (
    fraud_config.get(
        "ordinal_features",
        []
    )
)


categorical_features = (
    fraud_config.get(
        "categorical_features",
        []
    )
)


features = (
    continuous_features
    +
    ordinal_features
    +
    categorical_features
)


# ============================================================
# AFFICHAGE
# ============================================================

print(
    "\n=== CONTINUOUS FEATURES ==="
)

print(
    continuous_features
)


print(
    "\n=== ORDINAL FEATURES ==="
)

print(
    ordinal_features
)


print(
    "\n=== CATEGORICAL FEATURES ==="
)

for col in categorical_features:
    print(
        "-",
        col
    )


print(
    "\n=== TOTAL ==="
)

print(
    "Continuous :",
    len(continuous_features)
)

print(
    "Ordinal :",
    len(ordinal_features)
)

print(
    "Categorical :",
    len(categorical_features)
)

print(
    "Total model features :",
    len(features)
)

Feature manifest keys : ['claim_occurrence', 'claim_frequency', 'fraud_detection']

⚠️ Clé 'fraud' absente du feature_manifest.json.
Reconstruction explicite selon les décisions de Feature Engineering.

=== CONTINUOUS FEATURES ===
['Age', 'Deductible']

=== ORDINAL FEATURES ===
['WeekOfMonth', 'WeekOfMonthClaimed', 'DriverRating']

=== CATEGORICAL FEATURES ===
- Month
- DayOfWeek
- Make
- AccidentArea
- DayOfWeekClaimed
- MonthClaimed
- Sex
- MaritalStatus
- VehicleCategory
- VehiclePrice
- PastNumberOfClaims
- AgeOfVehicle
- AgeOfPolicyHolder
- AgentType
- NumberOfCars
- BasePolicy

=== TOTAL ===
Continuous : 2
Ordinal : 3
Categorical : 16
Total model features : 21


In [9]:
forbidden_features = {
    "PolicyNumber",
    "FraudFound_P",
    "Year",
    "RepNumber",
    "PolicyType",
    "Fault",
    "PoliceReportFiled",
    "WitnessPresent",
    "NumberOfSuppliments",
    "Days_Policy_Accident",
    "Days_Policy_Claim",
    "AddressChange_Claim"
}


unexpected_features = (
    forbidden_features
    .intersection(
        set(features)
    )
)


print(
    "Forbidden features present :",
    unexpected_features
)


assert len(unexpected_features) == 0, (
    "❌ Une ou plusieurs variables exclues "
    "sont présentes dans les features Fraud."
)


print(
    "✅ Leakage / exclusion check : OK"
)

Forbidden features present : set()
✅ Leakage / exclusion check : OK


In [10]:
print(
    "\n=== FEATURES UTILISÉES PAR LE MODÈLE ==="
)

for i, col in enumerate(
    features,
    start=1
):
    print(
        f"{i:02d}. {col}"
    )


print(
    "\nNombre total :",
    len(features)
)


=== FEATURES UTILISÉES PAR LE MODÈLE ===
01. Age
02. Deductible
03. WeekOfMonth
04. WeekOfMonthClaimed
05. DriverRating
06. Month
07. DayOfWeek
08. Make
09. AccidentArea
10. DayOfWeekClaimed
11. MonthClaimed
12. Sex
13. MaritalStatus
14. VehicleCategory
15. VehiclePrice
16. PastNumberOfClaims
17. AgeOfVehicle
18. AgeOfPolicyHolder
19. AgentType
20. NumberOfCars
21. BasePolicy

Nombre total : 21


In [11]:
continuous_features = (
    fraud_config.get(
        "continuous_features",
        fraud_config.get(
            "numeric_features",
            []
        )
    )
)

ordinal_features = (
    fraud_config.get(
        "ordinal_features",
        []
    )
)

categorical_features = (
    fraud_config[
        "categorical_features"
    ]
)


features = (
    continuous_features
    +
    ordinal_features
    +
    categorical_features
)


print(
    "Continuous :",
    continuous_features
)

print(
    "Ordinal :",
    ordinal_features
)

print(
    "Categorical :",
    categorical_features
)

print(
    "Total model features :",
    len(features)
)

Continuous : ['Age', 'Deductible']
Ordinal : ['WeekOfMonth', 'WeekOfMonthClaimed', 'DriverRating']
Categorical : ['Month', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'Sex', 'MaritalStatus', 'VehicleCategory', 'VehiclePrice', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'AgentType', 'NumberOfCars', 'BasePolicy']
Total model features : 21


In [12]:
forbidden_features = {
    "PolicyNumber",
    "FraudFound_P",
    "Year",
    "RepNumber",
    "PolicyType",
    "Fault",
    "PoliceReportFiled",
    "WitnessPresent",
    "NumberOfSuppliments",
    "Days_Policy_Accident",
    "Days_Policy_Claim",
    "AddressChange_Claim"
}


unexpected_features = (
    forbidden_features
    .intersection(
        set(features)
    )
)


print(
    "Forbidden features present :",
    unexpected_features
)


assert len(
    unexpected_features
) == 0, (
    "Des variables exclues/leakage sont présentes "
    "dans les features Fraud."
)

Forbidden features present : set()


In [13]:
fraud_train = (
    fraud[
        fraud["Year"] == 1994
    ]
    .copy()
)

fraud_validation = (
    fraud[
        fraud["Year"] == 1995
    ]
    .copy()
)

fraud_test = (
    fraud[
        fraud["Year"] == 1996
    ]
    .copy()
)


print(
    "TRAIN 1994 :",
    fraud_train.shape
)

print(
    "VALIDATION 1995 :",
    fraud_validation.shape
)

print(
    "TEST 1996 :",
    fraud_test.shape
)

TRAIN 1994 : (6142, 25)
VALIDATION 1995 : (5195, 25)
TEST 1996 : (4083, 25)


In [14]:
for name, df in [
    (
        "TRAIN 1994",
        fraud_train
    ),
    (
        "VALIDATION 1995",
        fraud_validation
    ),
    (
        "TEST 1996",
        fraud_test
    )
]:

    print(
        name,
        "->",
        len(df),
        "observations | fraud rate =",
        df["FraudFound_P"].mean()
    )

TRAIN 1994 -> 6142 observations | fraud rate = 0.06659068707261478
VALIDATION 1995 -> 5195 observations | fraud rate = 0.05794032723772859
TEST 1996 -> 4083 observations | fraud rate = 0.05216752387950037


In [15]:
X_train = (
    fraud_train[
        features
    ]
    .copy()
)

y_train = (
    fraud_train[
        "FraudFound_P"
    ]
    .astype(int)
    .copy()
)


X_validation = (
    fraud_validation[
        features
    ]
    .copy()
)

y_validation = (
    fraud_validation[
        "FraudFound_P"
    ]
    .astype(int)
    .copy()
)

In [16]:
train_prevalence = (
    y_train.mean()
)

validation_prevalence = (
    y_validation.mean()
)


print(
    "Train prevalence :",
    train_prevalence
)

print(
    "Validation prevalence :",
    validation_prevalence
)

Train prevalence : 0.06659068707261478
Validation prevalence : 0.05794032723772859


In [17]:
continuous_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [18]:
ordinal_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [19]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [20]:
linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            continuous_pipeline,
            continuous_features
        ),
        (
            "ordinal",
            ordinal_pipeline,
            ordinal_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [21]:
tree_continuous_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        )
    ]
)


tree_ordinal_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)


tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            tree_continuous_pipeline,
            continuous_features
        ),
        (
            "ordinal",
            tree_ordinal_pipeline,
            ordinal_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [22]:
n_negative = (
    y_train == 0
).sum()

n_positive = (
    y_train == 1
).sum()


class_ratio = (
    n_negative
    /
    n_positive
)


print(
    "Negative :",
    n_negative
)

print(
    "Positive :",
    n_positive
)

print(
    "Class ratio :",
    class_ratio
)

Negative : 5733
Positive : 409
Class ratio : 14.017114914425427


In [23]:
dummy_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            linear_preprocessor
        ),
        (
            "model",
            DummyClassifier(
                strategy="prior"
            )
        )
    ]
)

In [24]:
logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            linear_preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1500,
                random_state=RANDOM_STATE
            )
        )
    ]
)

In [25]:
logistic_balanced_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            linear_preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1500,
                class_weight="balanced",
                random_state=RANDOM_STATE
            )
        )
    ]
)

In [26]:
random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

In [27]:
xgb_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",

                n_estimators=300,
                learning_rate=0.05,
                max_depth=5,

                subsample=0.8,
                colsample_bytree=0.8,

                scale_pos_weight=float(
                    class_ratio
                ),

                tree_method="hist",

                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

In [28]:
def evaluate_fraud_model(
    model_name,
    pipeline,
    X_train,
    y_train,
    X_validation,
    y_validation
):

    start = (
        time.perf_counter()
    )


    pipeline.fit(
        X_train,
        y_train
    )


    elapsed = (
        time.perf_counter()
        -
        start
    )


    probabilities = (
        pipeline
        .predict_proba(
            X_validation
        )[:, 1]
    )


    predictions = (
        probabilities >= 0.5
    ).astype(int)


    result = {
        "model":
            model_name,

        "pr_auc":
            average_precision_score(
                y_validation,
                probabilities
            ),

        "roc_auc":
            roc_auc_score(
                y_validation,
                probabilities
            ),

        "precision":
            precision_score(
                y_validation,
                predictions,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_validation,
                predictions,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_validation,
                predictions,
                zero_division=0
            ),

        "brier":
            brier_score_loss(
                y_validation,
                probabilities
            ),

        "fit_time":
            elapsed
    }


    return (
        pipeline,
        probabilities,
        result
    )

In [29]:
models = {
    "dummy_prior":
        dummy_pipeline,

    "logistic":
        logistic_pipeline,

    "logistic_balanced":
        logistic_balanced_pipeline,

    "random_forest":
        random_forest_pipeline,

    "xgboost":
        xgb_pipeline
}

In [30]:
fraud_results = []

fitted_models = {}

validation_probabilities = {}


for name, pipeline in (
    models.items()
):

    print(
        "\nRunning :",
        name
    )


    fitted_model, probabilities, result = (
        evaluate_fraud_model(
            model_name=name,
            pipeline=pipeline,
            X_train=X_train,
            y_train=y_train,
            X_validation=X_validation,
            y_validation=y_validation
        )
    )


    fitted_models[
        name
    ] = fitted_model


    validation_probabilities[
        name
    ] = probabilities


    fraud_results.append(
        result
    )


    print(
        result
    )


Running : dummy_prior
{'model': 'dummy_prior', 'pr_auc': 0.05794032723772859, 'roc_auc': 0.5, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'brier': 0.054658074442586524, 'fit_time': 1.2602790999808349}

Running : logistic
{'model': 'logistic', 'pr_auc': 0.07108774230857878, 'roc_auc': 0.5948255847895654, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'brier': 0.05810276621246242, 'fit_time': 0.15174210001714528}

Running : logistic_balanced
{'model': 'logistic_balanced', 'pr_auc': 0.07149363217394417, 'roc_auc': 0.578360240419145, 'precision': 0.06727037516170763, 'recall': 0.34551495016611294, 'f1': 0.11261505143475907, 'brier': 0.18742603028276425, 'fit_time': 0.2532397999893874}

Running : random_forest
{'model': 'random_forest', 'pr_auc': 0.07603784423806499, 'roc_auc': 0.6251841362465668, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'brier': 0.056670839482408294, 'fit_time': 3.653316400013864}

Running : xgboost
{'model': 'xgboost', 'pr_auc': 0.07827258219019291, 'roc_auc': 0.58547994

In [31]:
fraud_comparison = (
    pd.DataFrame(
        fraud_results
    )
    .sort_values(
        "pr_auc",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


fraud_comparison.round(
    6
)

,model,pr_auc,roc_auc,precision,recall,f1,brier,fit_time
0,xgboost,0.078273,0.585480,0.072022,0.172757,0.101662,0.118390,2.935956
1,random_forest,0.076038,0.625184,0.000000,0.000000,0.000000,0.056671,3.653316
2,logistic_balanced,0.071494,0.578360,0.067270,0.345515,0.112615,0.187426,0.253240
3,logistic,0.071088,0.594826,0.000000,0.000000,0.000000,0.058103,0.151742
4,dummy_prior,0.057940,0.500000,0.000000,0.000000,0.000000,0.054658,1.260279


In [32]:
fraud_comparison.to_csv(
    REPORT_DIR
    / "fraud_initial_model_comparison.csv",
    index=False
)

In [33]:
print(
    "=" * 80
)

print(
    "FRAUD CLASSIFICATION — INITIAL MODEL RESULTS"
)

print(
    "=" * 80
)


print(
    "\n=== TEMPORAL SPLIT ==="
)

print(
    "1994 train size :",
    len(fraud_train)
)

print(
    "1995 validation size :",
    len(fraud_validation)
)

print(
    "1996 test size :",
    len(fraud_test)
)


print(
    "\n=== FRAUD RATES ==="
)

print(
    "1994 train fraud rate :",
    y_train.mean()
)

print(
    "1995 validation fraud rate :",
    y_validation.mean()
)

print(
    "1996 test fraud rate :",
    fraud_test[
        "FraudFound_P"
    ].mean()
)


print(
    "\n=== CLASS RATIO TRAIN ==="
)

print(
    class_ratio
)


print(
    "\n=== MODEL COMPARISON ==="
)

print(
    fraud_comparison
    .round(6)
    .to_string(
        index=False
    )
)


print(
    "\n=== PR-AUC VS VALIDATION PREVALENCE ==="
)

for _, row in (
    fraud_comparison.iterrows()
):

    print(
        row["model"],
        "-> PR-AUC =",
        round(
            row["pr_auc"],
            6
        ),
        "| lift =",
        round(
            row["pr_auc"]
            /
            validation_prevalence,
            3
        ),
        "x"
    )


print(
    "\n1996 FINAL TEST UTILISÉ POUR MODEL SELECTION ? NON"
)

print(
    "=" * 80
)

FRAUD CLASSIFICATION — INITIAL MODEL RESULTS

=== TEMPORAL SPLIT ===
1994 train size : 6142
1995 validation size : 5195
1996 test size : 4083

=== FRAUD RATES ===
1994 train fraud rate : 0.06659068707261478
1995 validation fraud rate : 0.05794032723772859
1996 test fraud rate : 0.05216752387950037

=== CLASS RATIO TRAIN ===
14.017114914425427

=== MODEL COMPARISON ===
            model   pr_auc  roc_auc  precision   recall       f1    brier  fit_time
          xgboost 0.078273 0.585480   0.072022 0.172757 0.101662 0.118390  2.935956
    random_forest 0.076038 0.625184   0.000000 0.000000 0.000000 0.056671  3.653316
logistic_balanced 0.071494 0.578360   0.067270 0.345515 0.112615 0.187426  0.253240
         logistic 0.071088 0.594826   0.000000 0.000000 0.000000 0.058103  0.151742
      dummy_prior 0.057940 0.500000   0.000000 0.000000 0.000000 0.054658  1.260279

=== PR-AUC VS VALIDATION PREVALENCE ===
xgboost -> PR-AUC = 0.078273 | lift = 1.351 x
random_forest -> PR-AUC = 0.076038 | l

# Analyse comparative — Fraud Classification

## Contexte

Le protocole expérimental est temporel :

- 1994 : entraînement ;
- 1995 : validation ;
- 1996 : test final.

Les taux de fraude observés sont :

- 1994 : 6.66 % ;
- 1995 : 5.79 % ;
- 1996 : 5.22 %.

Cette diminution indique une évolution temporelle du taux de base
de la fraude et justifie l'utilisation d'un protocole temporel.

Le jeu de test 1996 n'est pas utilisé pour sélectionner les modèles.

---

## DummyClassifier

PR-AUC :

**0.05794**

ROC-AUC :

**0.5000**

Le DummyClassifier représente la baseline minimale.

Sa PR-AUC correspond pratiquement à la prévalence de fraude
de l'année 1995.

Il ne possède donc aucun pouvoir discriminant.

---

## Logistic Regression

PR-AUC :

**0.07109**

ROC-AUC :

**0.59483**

Brier Score :

**0.05810**

Au seuil de 0.5 :

- Precision = 0 ;
- Recall = 0 ;
- F1 = 0.

Malgré l'absence de prédictions positives au seuil de 0.5,
la PR-AUC et le ROC-AUC restent supérieurs à la baseline.

Le problème provient donc notamment du seuil de classification,
qui est inadapté à une classe positive rare.

---

## Logistic Regression Balanced

PR-AUC :

**0.07149**

ROC-AUC :

**0.57836**

Precision :

**0.06727**

Recall :

**0.34552**

F1 :

**0.11262**

Brier Score :

**0.18743**

L'utilisation de `class_weight="balanced"` augmente fortement
le Recall mais dégrade considérablement la qualité des probabilités.

La PR-AUC reste très proche de celle de la régression logistique
non pondérée.

La pondération modifie donc principalement le comportement
de classification plutôt que le pouvoir de classement du modèle.

---

## Random Forest

PR-AUC :

**0.07604**

ROC-AUC :

**0.62518**

Brier Score :

**0.05667**

Random Forest obtient la deuxième meilleure PR-AUC et le meilleur
ROC-AUC parmi les modèles testés.

Au seuil de 0.5, aucune observation n'est cependant classée frauduleuse.

Ce résultat ne signifie pas que le modèle ne possède aucun signal :
sa PR-AUC et son ROC-AUC montrent qu'il parvient partiellement
à ordonner les dossiers selon leur risque.

Le seuil de 0.5 est simplement trop élevé pour les probabilités
produites dans ce contexte fortement déséquilibré.

---

## XGBoost

PR-AUC :

**0.07827**

ROC-AUC :

**0.58548**

Precision :

**0.07202**

Recall :

**0.17276**

F1 :

**0.10166**

Brier Score :

**0.11839**

XGBoost obtient la meilleure PR-AUC parmi les modèles évalués.

Sa PR-AUC est environ :

**1.35 fois la prévalence de fraude de la validation.**

Le modèle possède donc un signal prédictif réel,
mais celui-ci reste modéré.

La pondération importante de la classe positive permet d'améliorer
la détection de la fraude mais entraîne une forte dégradation
de la calibration des probabilités.

---

# Conclusion provisoire

Les deux modèles retenus pour l'étape suivante sont :

1. **XGBoost**
2. **Random Forest**

XGBoost est retenu principalement pour sa PR-AUC,
qui constitue la métrique principale du problème.

Random Forest est conservé car il présente une PR-AUC proche,
le meilleur ROC-AUC et un Brier Score beaucoup plus favorable.

Logistic Regression reste une baseline statistique de référence.

Les performances modestes doivent être interprétées dans le contexte
d'un dataset limité, d'une classe positive rare, d'un protocole temporel
strict et d'une sélection volontairement conservatrice des variables
afin d'éviter les risques de leakage.

Le test final 1996 reste fermé.